# 12-15 mag v2 candidate discrepancy audit

Visual notebook version of `scripts/audit_12_15_v2_candidate_discrepancies.py`. It runs the same audit logic, writes the same CSV/JSON reports when enabled, and adds charts plus drill-down tables for inspection.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/malca-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/malca-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

try:
    import seaborn as sns
except ImportError:
    sns = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")


def mag_bin_sort_key(value: object) -> tuple[float, str]:
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>", "null"}:
        return (float("inf"), text)
    start = text.split("_", 1)[0]
    try:
        return (float(start), text)
    except ValueError:
        return (float("inf"), text)


def sort_by_mag_bin(df: pd.DataFrame, mag_bin_col: str, *extra_cols: str) -> pd.DataFrame:
    sort_cols = ["_mag_bin_sort", *[col for col in extra_cols if col in df.columns]]
    return (
        df.assign(_mag_bin_sort=df[mag_bin_col].map(mag_bin_sort_key))
        .sort_values(sort_cols, kind="stable")
        .drop(columns="_mag_bin_sort")
        .reset_index(drop=True)
    )


def relative_to_repo(value: object) -> str:
    path = Path(str(value))
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(value)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCRIPT_PATH = REPO_ROOT / "scripts/audit_12_15_v2_candidate_discrepancies.py"
spec = importlib.util.spec_from_file_location("audit_12_15_v2_candidate_discrepancies", SCRIPT_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not import audit script from {SCRIPT_PATH}")
audit_mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = audit_mod
spec.loader.exec_module(audit_mod)

REPO_ROOT

## Inputs

In [ ]:
# Edit these if you want to inspect a different candidate file, reproduction run, or output directory.
CANDIDATES_CSV = REPO_ROOT / audit_mod.DEFAULT_CANDIDATES_CSV
REPRODUCTION_CSV = None  # None uses newest output/logs/reproduction/reproduction_*.csv.
OUTPUT_DIR = REPO_ROOT / audit_mod.DEFAULT_OUTPUT_DIR

ALLOW_UNMATCHED_REPRODUCTION = False
SKIP_PROVENANCE = False
PROVENANCE_TABLES = [REPO_ROOT / path for path in audit_mod.DEFAULT_PROVENANCE_TABLES]
WRITE_REPORTS = True

CANDIDATES_CSV, REPRODUCTION_CSV, OUTPUT_DIR

In [ ]:
def as_repo_path(path: str | Path | None) -> Path | None:
    if path is None:
        return None
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


candidates_path = as_repo_path(CANDIDATES_CSV)
output_dir = as_repo_path(OUTPUT_DIR)
reproduction_path = as_repo_path(REPRODUCTION_CSV)
if reproduction_path is None:
    reproduction_path = audit_mod.find_latest_reproduction_csv(REPO_ROOT / audit_mod.DEFAULT_REPRODUCTION_DIR)

provenance_tables = None
if not SKIP_PROVENANCE:
    provenance_tables = [as_repo_path(path) for path in PROVENANCE_TABLES]

candidates = audit_mod.load_primary_candidates(candidates_path)
reproduction = audit_mod.load_reproduction_results(reproduction_path)
audit = audit_mod.build_audit(
    candidates,
    reproduction,
    allow_unmatched_reproduction=ALLOW_UNMATCHED_REPRODUCTION,
    provenance_tables=provenance_tables,
)

if WRITE_REPORTS:
    audit["outputs"] = audit_mod.write_reports(audit, output_dir)
else:
    audit["outputs"] = {}

audit["candidates_csv"] = candidates_path
audit["reproduction_csv"] = reproduction_path

print(f"Candidates CSV: {audit['candidates_csv']}")
print(f"Reproduction CSV: {audit['reproduction_csv']}")
print(f"Output dir: {output_dir}")

summary = pd.Series(audit["summary"], name="value").rename_axis("metric").reset_index()
display(summary)

## Overview

In [ ]:
candidate_counts = sort_by_mag_bin(audit["candidate_counts_by_mag_bin"].copy(), "mag_bin")
overlap = sort_by_mag_bin(audit["brayden_overlap_by_expected_mag_bin"].copy(), "expected_mag_bin")
discrepancy_counts = audit["discrepancy_counts_by_category"].copy().sort_values("count")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

candidate_x = range(len(candidate_counts))
axes[0].bar(candidate_x, candidate_counts["n_candidates"], color="#4c78a8")
axes[0].set_title("Primary v2 candidates by mag bin")
axes[0].set_xlabel("mag_bin")
axes[0].set_ylabel("candidates")
axes[0].set_xticks(candidate_x, candidate_counts["mag_bin"])
axes[0].tick_params(axis="x", rotation=30)

overlap_x = range(len(overlap))
axes[1].bar(
    overlap_x,
    overlap["present_in_12_15_v2"],
    color="#59a14f",
    label="present in v2",
)
axes[1].bar(
    overlap_x,
    overlap["missing_from_12_15_v2"],
    bottom=overlap["present_in_12_15_v2"],
    color="#e15759",
    label="missing from v2",
)
for idx, row in overlap.reset_index(drop=True).iterrows():
    axes[1].text(idx, row["brayden_total"] + 0.25, f"{row['fraction_present']:.0%}", ha="center", va="bottom", fontsize=9)
axes[1].set_title("Brayden target overlap")
axes[1].set_xlabel("expected_mag_bin")
axes[1].set_ylabel("targets")
axes[1].set_xticks(overlap_x, overlap["expected_mag_bin"])
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(frameon=False)

axes[2].barh(discrepancy_counts["discrepancy_category"], discrepancy_counts["count"], color="#f28e2b")
axes[2].set_title("Discrepancy categories")
axes[2].set_xlabel("targets")
axes[2].set_ylabel("")

plt.show()


## Drill-down tables

In [ ]:
comparison = audit["comparison"].copy()
detail_cols = [
    "source",
    "source_id",
    "category",
    "expected_mag_bin",
    "v2_mag_bin",
    "expected_detected",
    "reproduction_detected",
    "reproduction_rejection_reason",
    "reproduction_detection_details",
    "in_12_15_v2",
    "discrepancy_category",
    "g_bayes_dip_bayes_factor",
    "v_bayes_dip_bayes_factor",
    "g_n_runs",
    "v_n_runs",
]
detail_cols = [column for column in detail_cols if column in comparison.columns]

missing_from_v2 = sort_by_mag_bin(
    comparison.loc[~comparison["in_12_15_v2"], detail_cols], "expected_mag_bin", "source_id"
)
mag_bin_mismatches = sort_by_mag_bin(audit["mag_bin_mismatches"].copy(), "expected_mag_bin", "source_id")
present_but_rejected = sort_by_mag_bin(
    comparison.loc[comparison["in_12_15_v2"] & comparison["reproduction_detected"].eq(False), detail_cols],
    "expected_mag_bin",
    "source_id",
)

display(Markdown(f"### Missing from v2 ({len(missing_from_v2)})"))
display(missing_from_v2)

display(Markdown(f"### Mag-bin mismatches ({len(mag_bin_mismatches)})"))
display(mag_bin_mismatches[detail_cols])

display(Markdown(f"### Present in v2 but rejected in reproduction ({len(present_but_rejected)})"))
display(present_but_rejected)

## In-v2 Candidate Panels

In [ ]:
# Set this to a specific source_id to inspect a different in-v2 candidate panel.
# None selects the first non-consistent in-v2 candidate with an available PNG panel.
PANEL_SOURCE_ID = None


def latest_reproduction_panel_dir(path: Path) -> Path:
    run_name = Path(path).stem
    if run_name.startswith("reproduction_"):
        run_name = run_name[len("reproduction_") :]
    return REPO_ROOT / "output/plots/reproduction" / run_name


PANEL_SEARCH_DIRS = [
    REPO_ROOT / "output/plots",
    latest_reproduction_panel_dir(reproduction_path),
    REPO_ROOT / "output/plots/skypatrol_gp_masked",
    REPO_ROOT / "output/plots/skypatrol_gp",
]


def candidate_panel_path(source_id: object, *, prefer_png: bool = True) -> Path | None:
    sid = str(source_id).strip()
    if not sid:
        return None
    suffixes = ["_dips.png", "_dips.pdf"] if prefer_png else ["_dips.pdf", "_dips.png"]
    for suffix in suffixes:
        for directory in PANEL_SEARCH_DIRS:
            path = directory / f"{sid}{suffix}"
            if path.is_file():
                return path
    return None


candidate_lookup = candidates[
    [column for column in ["asas_sn_id", "asas_sn_id_norm", "mag_bin", "path", "ra_deg", "dec_deg"] if column in candidates.columns]
].rename(
    columns={
        "asas_sn_id": "v2_asas_sn_id",
        "mag_bin": "candidate_csv_mag_bin",
        "path": "candidate_csv_path",
    }
)
in_v2_panels = comparison.loc[comparison["in_12_15_v2"], detail_cols].merge(
    candidate_lookup,
    left_on="source_id",
    right_on="v2_asas_sn_id",
    how="left",
    validate="one_to_one",
)
in_v2_panels["panel_path"] = in_v2_panels["source_id"].map(candidate_panel_path)
in_v2_panels["panel_available"] = in_v2_panels["panel_path"].notna()
in_v2_panels["panel_path"] = in_v2_panels["panel_path"].map(lambda path: relative_to_repo(path) if path is not None else "")
in_v2_panels = sort_by_mag_bin(in_v2_panels, "expected_mag_bin", "discrepancy_category", "source_id")

panel_table_cols = [
    "source",
    "source_id",
    "category",
    "expected_mag_bin",
    "v2_mag_bin",
    "discrepancy_category",
    "panel_available",
    "panel_path",
    "candidate_csv_path",
]
panel_table_cols = [column for column in panel_table_cols if column in in_v2_panels.columns]
display(Markdown(f"### In-v2 Brayden candidates ({len(in_v2_panels)})"))
display(in_v2_panels[panel_table_cols])

available = in_v2_panels.loc[in_v2_panels["panel_available"]].copy()
if PANEL_SOURCE_ID is not None:
    selected = in_v2_panels.loc[in_v2_panels["source_id"].astype(str).eq(str(PANEL_SOURCE_ID))]
else:
    selected = available.loc[available["discrepancy_category"].ne("consistent_present_detected")].head(1)
    if selected.empty:
        selected = available.head(1)

if selected.empty:
    print("No available panel PNG/PDF was found for the selected in-v2 candidates.")
else:
    selected_row = selected.iloc[0]
    panel_path = candidate_panel_path(selected_row["source_id"])
    display(Markdown(f"### Panel: `{selected_row['source']}` / `{selected_row['source_id']}`"))
    display(selected_row[[column for column in detail_cols if column in selected_row.index]].to_frame("value"))
    if panel_path is not None and panel_path.suffix.lower() == ".png":
        display(Image(filename=str(panel_path), width=1250))
    elif panel_path is not None:
        display(Markdown(f"Panel PDF: `{relative_to_repo(panel_path)}`"))
    else:
        print(f"No panel file found for source_id={selected_row['source_id']}.")

def show_candidate_panel(source_id: object):
    path = candidate_panel_path(source_id)
    if path is None:
        raise FileNotFoundError(f"No panel PNG/PDF found for source_id={source_id}")
    if path.suffix.lower() == ".png":
        display(Image(filename=str(path), width=1250))
    else:
        display(Markdown(f"Panel PDF: `{relative_to_repo(path)}`"))
    return path


## Dipper Candidates

In [ ]:
# Set this to a specific Dipper source_id to inspect that panel.
# None selects the first non-consistent in-v2 Dipper with an available panel.
DIPPER_PANEL_SOURCE_ID = None

dipper_mask = comparison["category"].astype("string").str.casefold().eq("dippers")
dipper_candidates = sort_by_mag_bin(
    comparison.loc[dipper_mask, detail_cols], "expected_mag_bin", "source_id"
)

display(Markdown(f"### Dipper audit rows ({len(dipper_candidates)})"))
display(dipper_candidates)

if dipper_candidates.empty:
    print("No Dipper candidates found in the Brayden comparison table.")
else:
    dipper_by_mag = pd.crosstab(dipper_candidates["expected_mag_bin"], dipper_candidates["discrepancy_category"])
    dipper_by_mag = dipper_by_mag.reindex(sorted(dipper_by_mag.index, key=mag_bin_sort_key)).fillna(0).astype(int)
    display(Markdown("### Dipper discrepancy category by expected mag bin"))
    display(dipper_by_mag)

    dipper_status = pd.crosstab(
        [dipper_candidates["expected_detected"].astype(str), dipper_candidates["reproduction_detected"].astype(str)],
        dipper_candidates["in_12_15_v2"].map({True: "in_v2", False: "missing_from_v2"}),
        rownames=["expected_detected", "reproduction_detected"],
        colnames=["candidate_status"],
    )
    display(Markdown("### Dipper detection status by v2 membership"))
    display(dipper_status)

    dipper_panels = in_v2_panels.loc[
        in_v2_panels["category"].astype("string").str.casefold().eq("dippers")
    ].copy()
    dipper_panels = sort_by_mag_bin(dipper_panels, "expected_mag_bin", "discrepancy_category", "source_id")
    display(Markdown(f"### In-v2 Dipper panels ({len(dipper_panels)})"))
    display(dipper_panels[[column for column in panel_table_cols if column in dipper_panels.columns]])

    available_dipper_panels = dipper_panels.loc[dipper_panels["panel_available"]].copy()
    if DIPPER_PANEL_SOURCE_ID is not None:
        selected_dipper = dipper_panels.loc[dipper_panels["source_id"].astype(str).eq(str(DIPPER_PANEL_SOURCE_ID))]
    else:
        selected_dipper = available_dipper_panels.loc[
            available_dipper_panels["discrepancy_category"].ne("consistent_present_detected")
        ].head(1)
        if selected_dipper.empty:
            selected_dipper = available_dipper_panels.head(1)

    if selected_dipper.empty:
        print("No available Dipper panel PNG/PDF was found.")
    else:
        dipper_row = selected_dipper.iloc[0]
        dipper_panel_path = candidate_panel_path(dipper_row["source_id"])
        display(Markdown(f"### Dipper panel: `{dipper_row['source']}` / `{dipper_row['source_id']}`"))
        display(dipper_row[[column for column in detail_cols if column in dipper_row.index]].to_frame("value"))
        if dipper_panel_path is not None and dipper_panel_path.suffix.lower() == ".png":
            display(Image(filename=str(dipper_panel_path), width=1250))
        elif dipper_panel_path is not None:
            display(Markdown(f"Panel PDF: `{relative_to_repo(dipper_panel_path)}`"))
        else:
            print(f"No panel file found for source_id={dipper_row['source_id']}.")


## Reproduction/v2 Disagreements

In [ ]:
import pyarrow.parquet as pq

MAY_RUN_DIRS_BY_MAG_BIN = {
    "12_12.5": REPO_ROOT / "output/runs/output_bundle_12_12.5_home_bundle_12_12.5",
    "12.5_13": REPO_ROOT / "output/runs/output_bundle_12.5_13_home_bundle_12.5_13",
    "13_13.5": REPO_ROOT / "output/runs/output_bundle_13_13.5_bundle_13_13.5",
    "13.5_14": REPO_ROOT / "output/runs/output_bundle_13.5_14_bundle_13.5_14",
    "14_14.5": REPO_ROOT / "output/runs/output_bundle_14_14.5_bundle_14_14.5",
    "14.5_15": REPO_ROOT / "output/runs/output_bundle_14.5_15_bundle_14.5_15",
}

MAY_FILTER_COLUMNS = [
    "path",
    "dip_significant",
    "jump_significant",
    "dip_count",
    "jump_count",
    "dip_run_count",
    "jump_run_count",
    "dip_max_run_points",
    "jump_max_run_points",
    "dip_max_run_cameras",
    "jump_max_run_cameras",
    "dip_bayes_factor",
    "jump_bayes_factor",
    "dip_max_log_bf_local",
    "jump_max_log_bf_local",
    "dipper_score",
    "jumper_score",
    "failed_posterior_strength",
    "failed_significant_detection",
    "failed_run_robustness",
    "failed_morphology",
    "failed_score",
    "failed_periodic_catalog",
    "failed_gaia_ruwe",
    "failed_gaia_pm",
    "failed_periodicity",
    "failed_any",
    "catalog_match",
    "catalog_source",
    "period_primary_source",
    "period_source_periods",
    "periodic_flag",
    "periodicity_score",
    "high_ruwe_flag",
    "ruwe",
    "high_pm_flag",
    "pm_total",
    "phase_period_days",
    "phase_source",
]

MAY_MIN_BAYES_FACTOR = 10.0
MAY_MIN_RUN_COUNT = 1
MAY_MIN_RUN_POINTS = 2
MAY_MIN_RUN_CAMERAS = 2
MAY_MIN_SCORE = 0.0


def _boolish(value: object) -> bool:
    if isinstance(value, bool):
        return value
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().casefold() in {"1", "true", "t", "yes", "y"}


def _num(value: object) -> float:
    out = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    return float(out) if pd.notna(out) else float("nan")


def _fmt_num(value: object) -> str:
    number = _num(value)
    if pd.isna(number):
        return "NA"
    return f"{number:.3g}"


def _scan_may_parquet(path: Path, source_ids: set[str], columns: list[str]) -> pd.DataFrame:
    if not path.is_file() or not source_ids:
        return pd.DataFrame()
    parquet_file = pq.ParquetFile(path)
    available = [column for column in columns if column in parquet_file.schema.names]
    if "path" not in available:
        return pd.DataFrame()
    hits = []
    for batch in parquet_file.iter_batches(batch_size=250_000, columns=available):
        chunk = batch.to_pandas()
        source = chunk["path"].astype(str).str.extract(r"([^/]+)\.[^.\\/]+$", expand=False)
        mask = source.isin(source_ids)
        if mask.any():
            out = chunk.loc[mask].copy()
            out.insert(0, "source_id", source.loc[mask].to_numpy())
            hits.append(out)
    return pd.concat(hits, ignore_index=True) if hits else pd.DataFrame()


def _scan_may_stage(stage_file: str, source_ids: set[str], columns: list[str]) -> pd.DataFrame:
    frames = []
    for mag_bin, run_dir in MAY_RUN_DIRS_BY_MAG_BIN.items():
        path = run_dir / "results" / stage_file
        hits = _scan_may_parquet(path, source_ids, columns)
        if hits.empty:
            continue
        hits.insert(0, "may_run_mag_bin", mag_bin)
        hits.insert(0, "may_run_dir", relative_to_repo(run_dir))
        frames.append(hits)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _failed_filters(row: pd.Series) -> str:
    labels = []
    for column in row.index:
        if column.startswith("failed_") and column != "failed_any" and _boolish(row.get(column)):
            labels.append(column.removeprefix("failed_"))
    return ", ".join(labels)


def _run_robustness_detail(row: pd.Series) -> str:
    branch_details = []
    for branch in ["dip", "jump"]:
        reasons = []
        run_count = _num(row.get(f"{branch}_run_count"))
        run_points = _num(row.get(f"{branch}_max_run_points"))
        run_cameras = _num(row.get(f"{branch}_max_run_cameras"))
        if pd.isna(run_count) or run_count < MAY_MIN_RUN_COUNT:
            reasons.append(f"run_count={_fmt_num(run_count)}<{MAY_MIN_RUN_COUNT}")
        if pd.isna(run_points) or run_points < MAY_MIN_RUN_POINTS:
            reasons.append(f"max_run_points={_fmt_num(run_points)}<{MAY_MIN_RUN_POINTS}")
        if pd.isna(run_cameras) or run_cameras < MAY_MIN_RUN_CAMERAS:
            reasons.append(f"max_run_cameras={_fmt_num(run_cameras)}<{MAY_MIN_RUN_CAMERAS}")
        if reasons:
            branch_details.append(f"{branch}: " + ", ".join(reasons))
        else:
            branch_details.append(f"{branch}: passes run robustness")
    return "; ".join(branch_details)


def _significant_detection_detail(row: pd.Series) -> str:
    return (
        "requires significant flag plus >=1 peak/run; "
        f"dip_sig={_boolish(row.get('dip_significant'))}, dip_count={_fmt_num(row.get('dip_count'))}, "
        f"dip_run_count={_fmt_num(row.get('dip_run_count'))}; "
        f"jump_sig={_boolish(row.get('jump_significant'))}, jump_count={_fmt_num(row.get('jump_count'))}, "
        f"jump_run_count={_fmt_num(row.get('jump_run_count'))}"
    )


def _posterior_strength_detail(row: pd.Series) -> str:
    return (
        f"requires dip/jump Bayes factor > {MAY_MIN_BAYES_FACTOR:g} and finite local BF; "
        f"dip_bayes_factor={_fmt_num(row.get('dip_bayes_factor'))}, "
        f"jump_bayes_factor={_fmt_num(row.get('jump_bayes_factor'))}, "
        f"dip_max_log_bf_local={_fmt_num(row.get('dip_max_log_bf_local'))}, "
        f"jump_max_log_bf_local={_fmt_num(row.get('jump_max_log_bf_local'))}"
    )


def _score_detail(row: pd.Series) -> str:
    return (
        f"requires dipper_score or jumper_score >= {MAY_MIN_SCORE:g}; "
        f"dipper_score={_fmt_num(row.get('dipper_score'))}, jumper_score={_fmt_num(row.get('jumper_score'))}"
    )


def _may_nondetection_reason(row: pd.Series) -> str:
    if not _boolish(row.get("may_results_present")):
        return "not found in scanned May/home run event-results tables"
    if not _boolish(row.get("may_filtered_present")):
        return "present in May event results, but not found in May post-filter audit table"
    failed = str(row.get("may_failed_filters") or "").strip()
    if not failed:
        if _boolish(row.get("may_enriched_present")):
            return "present in May enriched/final candidate artifact"
        return "passes May post-filter audit, but absent from enriched/final candidate artifact; check downstream enrichment/export"
    details = []
    if "posterior_strength" in failed:
        details.append("posterior_strength: " + _posterior_strength_detail(row))
    if "significant_detection" in failed:
        details.append("significant_detection: " + _significant_detection_detail(row))
    if "run_robustness" in failed:
        details.append("run_robustness: " + _run_robustness_detail(row))
    if "score" in failed:
        details.append("score: " + _score_detail(row))
    for label in ["morphology", "periodic_catalog", "gaia_ruwe", "gaia_pm", "periodicity"]:
        if label in failed:
            details.append(f"{label}: failed_{label}=True in May post-filter audit")
    return " | ".join(details) if details else f"failed May filters: {failed}"


def build_may_run_diagnostics(source_ids: pd.Series) -> pd.DataFrame:
    ids = {str(value).strip() for value in source_ids if str(value).strip()}
    base = pd.DataFrame({"source_id": sorted(ids)})
    results = _scan_may_stage("lc_events_results.parquet", ids, ["path"])
    filtered = _scan_may_stage("lc_events_filtered.parquet", ids, MAY_FILTER_COLUMNS)
    enriched = _scan_may_stage("lc_events_enriched.parquet", ids, ["path"])

    result_bins = results.groupby("source_id")["may_run_mag_bin"].agg(lambda values: ", ".join(sorted(set(values)))) if not results.empty else pd.Series(dtype=str)
    enriched_bins = enriched.groupby("source_id")["may_run_mag_bin"].agg(lambda values: ", ".join(sorted(set(values)))) if not enriched.empty else pd.Series(dtype=str)

    out = base.copy()
    out["may_results_present"] = out["source_id"].isin(set(results.get("source_id", [])))
    out["may_results_mag_bins"] = out["source_id"].map(result_bins).fillna("")
    out["may_enriched_present"] = out["source_id"].isin(set(enriched.get("source_id", [])))
    out["may_enriched_mag_bins"] = out["source_id"].map(enriched_bins).fillna("")

    if not filtered.empty:
        filtered = filtered.drop_duplicates("source_id", keep="first").copy()
        filtered["may_filtered_present"] = True
        filtered["may_filtered_failed_any"] = filtered.get("failed_any", False).map(_boolish) if "failed_any" in filtered.columns else False
        filtered["may_failed_filters"] = filtered.apply(_failed_filters, axis=1)
        keep_cols = [
            "source_id",
            "may_run_dir",
            "may_run_mag_bin",
            "path",
            "may_filtered_present",
            "may_filtered_failed_any",
            "may_failed_filters",
            *[column for column in MAY_FILTER_COLUMNS if column != "path" and column in filtered.columns],
        ]
        filtered = filtered[keep_cols].rename(columns={"path": "may_run_path"})
        out = out.merge(filtered, on="source_id", how="left")
    else:
        out["may_filtered_present"] = False
        out["may_filtered_failed_any"] = False
        out["may_failed_filters"] = ""

    out["may_filtered_present"] = out["may_filtered_present"].fillna(False).map(_boolish)
    out["may_filtered_failed_any"] = out["may_filtered_failed_any"].fillna(False).map(_boolish)
    out["may_failed_filters"] = out["may_failed_filters"].fillna("")
    out["may_nondetection_reason"] = out.apply(_may_nondetection_reason, axis=1)
    return out


may_run_diagnostics = build_may_run_diagnostics(comparison["source_id"])
may_reason_cols = [
    "source_id",
    "may_results_present",
    "may_results_mag_bins",
    "may_filtered_present",
    "may_filtered_failed_any",
    "may_failed_filters",
    "may_enriched_present",
    "may_enriched_mag_bins",
    "may_run_mag_bin",
    "may_nondetection_reason",
]
display(Markdown("### May/home run diagnostics for Brayden targets"))
display(may_run_diagnostics[[column for column in may_reason_cols if column in may_run_diagnostics.columns]])


In [ ]:
# Rows where the reproduction run says detected, but the candidate is absent from v2,
# or the reproduction run says rejected, but the candidate is present in v2.
DISAGREEMENT_PANEL_SOURCE_ID = None

disagreement_mask = comparison["reproduction_detected"].astype(bool).ne(comparison["in_12_15_v2"].astype(bool))
reproduction_v2_disagreements = comparison.loc[disagreement_mask, detail_cols].merge(
    may_run_diagnostics,
    on="source_id",
    how="left",
    validate="one_to_one",
)
reproduction_v2_disagreements = sort_by_mag_bin(
    reproduction_v2_disagreements, "expected_mag_bin", "category", "source_id"
)
dipper_reproduction_v2_disagreements = sort_by_mag_bin(
    reproduction_v2_disagreements.loc[
        reproduction_v2_disagreements["category"].astype("string").str.casefold().eq("dippers")
    ],
    "expected_mag_bin",
    "source_id",
)

display(Markdown(f"### All reproduction/v2 disagreements ({len(reproduction_v2_disagreements)})"))
display(reproduction_v2_disagreements)

display(Markdown(f"### Dipper reproduction/v2 disagreements ({len(dipper_reproduction_v2_disagreements)})"))
display(dipper_reproduction_v2_disagreements)

if not reproduction_v2_disagreements.empty:
    disagreement_summary = pd.crosstab(
        reproduction_v2_disagreements["category"],
        reproduction_v2_disagreements["discrepancy_category"],
    )
    display(Markdown("### Disagreement category by candidate class"))
    display(disagreement_summary)

    disagreement_panels = reproduction_v2_disagreements.copy()
    disagreement_panels["panel_path"] = disagreement_panels["source_id"].map(candidate_panel_path)
    disagreement_panels["panel_available"] = disagreement_panels["panel_path"].notna()
    disagreement_panels["panel_path"] = disagreement_panels["panel_path"].map(
        lambda path: relative_to_repo(path) if path is not None else ""
    )
    panel_cols = [
        "source",
        "source_id",
        "category",
        "expected_mag_bin",
        "v2_mag_bin",
        "reproduction_detected",
        "in_12_15_v2",
        "discrepancy_category",
        "may_filtered_failed_any",
        "may_failed_filters",
        "may_nondetection_reason",
        "panel_available",
        "panel_path",
    ]
    display(Markdown("### Disagreement panel lookup"))
    display(disagreement_panels[[col for col in panel_cols if col in disagreement_panels.columns]])

    available_disagreement_panels = disagreement_panels.loc[disagreement_panels["panel_available"]].copy()
    if DISAGREEMENT_PANEL_SOURCE_ID is not None:
        selected_disagreement = disagreement_panels.loc[
            disagreement_panels["source_id"].astype(str).eq(str(DISAGREEMENT_PANEL_SOURCE_ID))
        ]
    else:
        selected_disagreement = available_disagreement_panels.loc[
            available_disagreement_panels["category"].astype("string").str.casefold().eq("dippers")
        ].head(1)
        if selected_disagreement.empty:
            selected_disagreement = available_disagreement_panels.head(1)

    if selected_disagreement.empty:
        print("No available panel PNG/PDF was found for disagreement rows.")
    else:
        disagreement_row = selected_disagreement.iloc[0]
        disagreement_panel_path = candidate_panel_path(disagreement_row["source_id"])
        display(Markdown(f"### Disagreement panel: `{disagreement_row['source']}` / `{disagreement_row['source_id']}`"))
        selected_detail_cols = [
            *detail_cols,
            "may_results_present",
            "may_results_mag_bins",
            "may_filtered_present",
            "may_filtered_failed_any",
            "may_failed_filters",
            "may_enriched_present",
            "may_enriched_mag_bins",
            "may_run_mag_bin",
            "may_run_path",
            "may_nondetection_reason",
        ]
        display(disagreement_row[[column for column in selected_detail_cols if column in disagreement_row.index]].to_frame("value"))
        if disagreement_panel_path is not None and disagreement_panel_path.suffix.lower() == ".png":
            display(Image(filename=str(disagreement_panel_path), width=1250))
        elif disagreement_panel_path is not None:
            display(Markdown(f"Panel PDF: `{relative_to_repo(disagreement_panel_path)}`"))
        else:
            print(f"No panel file found for source_id={disagreement_row['source_id']}.")


## Category views

In [ ]:
status_counts = pd.crosstab(
    [comparison["expected_detected"].astype(str), comparison["reproduction_detected"].astype(str)],
    comparison["in_12_15_v2"].map({True: "in_v2", False: "missing_from_v2"}),
    rownames=["expected_detected", "reproduction_detected"],
    colnames=["candidate_status"],
)
display(Markdown("### Detection status by v2 membership"))
display(status_counts)

category_by_mag = pd.crosstab(comparison["expected_mag_bin"], comparison["discrepancy_category"])
category_by_mag = category_by_mag.reindex(sorted(category_by_mag.index, key=mag_bin_sort_key))
display(Markdown("### Discrepancy category by expected mag bin"))
display(category_by_mag)

fig_width = max(8, 0.95 * len(category_by_mag.columns))
fig, ax = plt.subplots(figsize=(fig_width, 3.8))
if sns is not None:
    sns.heatmap(category_by_mag, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
else:
    image = ax.imshow(category_by_mag.to_numpy(), cmap="Blues")
    ax.set_xticks(range(len(category_by_mag.columns)), category_by_mag.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(category_by_mag.index)), category_by_mag.index)
    for y, (_, row) in enumerate(category_by_mag.iterrows()):
        for x, value in enumerate(row):
            ax.text(x, y, str(value), ha="center", va="center")
ax.set_title("Discrepancy category by expected mag bin")
ax.set_xlabel("")
ax.set_ylabel("expected_mag_bin")
plt.show()

## Reproduction metrics

In [ ]:
metric_cols = [
    "g_bayes_dip_significant",
    "v_bayes_dip_significant",
    "g_bayes_dip_bayes_factor",
    "v_bayes_dip_bayes_factor",
    "g_n_runs",
    "v_n_runs",
]
available_metric_cols = [column for column in metric_cols if column in comparison.columns]
metric_df = comparison.copy()
for column in ["g_bayes_dip_bayes_factor", "v_bayes_dip_bayes_factor", "g_n_runs", "v_n_runs"]:
    if column in metric_df.columns:
        metric_df[column] = pd.to_numeric(metric_df[column], errors="coerce")

if available_metric_cols:
    display(metric_df[["source_id", "discrepancy_category", *available_metric_cols]].sort_values("discrepancy_category"))
else:
    print("No reproduction metric columns were found in this CSV.")

x_col = "g_bayes_dip_bayes_factor"
y_col = "v_bayes_dip_bayes_factor"
if x_col in metric_df.columns and y_col in metric_df.columns:
    plot_df = metric_df.dropna(subset=[x_col, y_col]).copy()
    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        for category, group in plot_df.groupby("discrepancy_category", sort=True):
            ax.scatter(group[x_col], group[y_col], s=70, alpha=0.85, label=category)
        if (plot_df[x_col] > 0).all() and (plot_df[y_col] > 0).all():
            ax.set_xscale("log")
            ax.set_yscale("log")
        else:
            ax.set_xscale("symlog")
            ax.set_yscale("symlog")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title("Reproduction Bayes factors by discrepancy category")
        ax.legend(frameon=False, bbox_to_anchor=(1.04, 1), loc="upper left")
        plt.show()
    else:
        print("Bayes factor columns exist, but all plotted values are missing.")

## Secondary provenance

In [ ]:
def relative_to_repo(value: object) -> str:
    path = Path(str(value))
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(value)


provenance = audit["secondary_parquet_provenance"].copy()
if provenance.empty:
    print("No secondary provenance rows were collected.")
else:
    provenance["provenance_table"] = provenance["provenance_table"].map(relative_to_repo)
    provenance_summary = (
        provenance.groupby(["provenance_table", "provenance_status"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["provenance_status", "rows", "provenance_table"], ascending=[True, False, True])
    )
    display(provenance_summary)

    present_summary = provenance_summary.loc[provenance_summary["provenance_status"].eq("present")]
    if not present_summary.empty:
        fig, ax = plt.subplots(figsize=(9, max(2.5, 0.45 * len(present_summary))))
        ax.barh(present_summary["provenance_table"], present_summary["rows"], color="#76b7b2")
        ax.set_xlabel("Brayden IDs present")
        ax.set_ylabel("")
        ax.set_title("Secondary parquet provenance")
        plt.show()

    display(provenance.sort_values(["source_id", "provenance_table"], kind="stable"))

## Report files

In [ ]:
outputs = audit.get("outputs", {})
if outputs:
    output_rows = [
        {"artifact": name, "path": relative_to_repo(path)}
        for name, path in sorted(outputs.items())
    ]
    display(pd.DataFrame(output_rows))
else:
    print("WRITE_REPORTS is False, so no report files were written from this notebook run.")